In [1]:
from plotly.io import show
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.cluster import HierarchicalClustering, LinkageMethod
from skfolio.datasets import load_sp500_dataset
from skfolio.distance import KendallDistance
from skfolio.optimization import (
    EqualWeighted,
    HierarchicalEqualRiskContribution,
)
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()

X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.5, shuffle=False)

In [2]:
model1 = HierarchicalEqualRiskContribution(
    risk_measure=RiskMeasure.CDAR, portfolio_params=dict(name="HERC-CDaR-Ward-Pearson")
)
model1.fit(X_train)
model1.weights_

array([0.05814153, 0.04927288, 0.09420898, 0.05726203, 0.02409012,
       0.08083036, 0.08070112, 0.02983878, 0.06656721, 0.02730592,
       0.01757816, 0.01140416, 0.0933616 , 0.04726087, 0.02170548,
       0.02428936, 0.00827991, 0.02575021, 0.14895433, 0.03319698])

In [3]:
ptf1 = model1.predict(X_train)
ptf1.plot_contribution(measure=RiskMeasure.CDAR)

In [4]:
fig = model1.hierarchical_clustering_estimator_.plot_dendrogram(heatmap=False)
show(fig)

In [5]:
model1.hierarchical_clustering_estimator_.plot_dendrogram()

In [6]:
model2 = HierarchicalEqualRiskContribution(
    risk_measure=RiskMeasure.CDAR,
    hierarchical_clustering_estimator=HierarchicalClustering(
        linkage_method=LinkageMethod.SINGLE,
    ),
    portfolio_params=dict(name="HERC-CDaR-Single-Pearson"),
)
model2.fit(X_train)
model2.hierarchical_clustering_estimator_.plot_dendrogram(heatmap=True)

In [7]:
ptf2 = model2.predict(X_train)
ptf2.plot_contribution(measure=RiskMeasure.CDAR)

In [8]:
model3 = HierarchicalEqualRiskContribution(
    risk_measure=RiskMeasure.CDAR,
    distance_estimator=KendallDistance(absolute=True),
    portfolio_params=dict(name="HERC-CDaR-Ward-Kendal"),
)
model3.fit(X_train)
model3.hierarchical_clustering_estimator_.plot_dendrogram(heatmap=True)

In [9]:
bench = EqualWeighted()
bench.fit(X_train)
bench.weights_

array([0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05,
       0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])

In [10]:
population_test = Population([])
for model in [model1, model2, model3, bench]:
    population_test.append(model.predict(X_test))

population_test.plot_cumulative_returns()

In [11]:
population_test.plot_composition()